# 节点 03 · 反向传播（1986）— 手撕两层网络解决 XOR

> 对应文档：[docs/03-backprop-1986.md](../docs/03-backprop-1986.md)

**目标**：从零用 NumPy 实现一个两层神经网络，用反向传播训练它，让它学会 XOR。

**环境依赖**：`numpy`, `matplotlib`（标准库，无需额外安装）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# notebook 从 notebooks/ 目录执行，assets 在 ../docs/assets/
os.makedirs('../docs/assets', exist_ok=True)

# 固定随机种子，保证可复现
np.random.seed(42)
print('NumPy version:', np.__version__)

## 1. XOR 数据集

| x1 | x2 | XOR |
|----|----|----- |
| 0  | 0  |  0  |
| 0  | 1  |  1  |
| 1  | 0  |  1  |
| 1  | 1  |  0  |

这是单层感知机永远搞不定的问题（节点02证明了这一点）。

In [ ]:
# XOR 输入：shape (4, 2)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)

# XOR 标签：shape (4, 1)
y = np.array([[0], [1], [1], [0]], dtype=float)

print('输入 X:')
print(X)
print('标签 y:', y.T)

## 2. 激活函数：Sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

它的导数有个漂亮的性质：$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$

**为什么用 sigmoid？**
- 输出在 (0, 1) 之间，适合表示概率
- 处处可导，链式法则可以工作
- 1986 年代最流行的选择（后来 ReLU 取代了它，见节点04）

In [ ]:
def sigmoid(z):
    """sigmoid 激活函数"""
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_deriv(z):
    """sigmoid 导数：σ(z) * (1 - σ(z))"""
    s = sigmoid(z)
    return s * (1 - s)

# 可视化 sigmoid 和它的导数
z_vals = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(z_vals, sigmoid(z_vals), color='steelblue', linewidth=2)
axes[0].set_title('sigmoid(z)')
axes[0].set_xlabel('z')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(z_vals, sigmoid_deriv(z_vals), color='coral', linewidth=2)
axes[1].set_title("sigmoid'(z) — 最大值只有 0.25")
axes[1].set_xlabel('z')
axes[1].axhline(0.25, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sigmoid 函数及其导数', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/assets/03_sigmoid.png', dpi=100, bbox_inches='tight')
plt.show()
print('注意导数最大值只有 0.25 — 这是梯度消失的根源')

## 3. 前向传播（Forward Pass）

网络结构：**输入层(2) → 隐藏层(4) → 输出层(1)**

```
x (2,) → z1 = W1·x + b1 → h = σ(z1) → z2 = W2·h + b2 → ŷ = σ(z2)
```

前向传播就是：把输入一步步往前算，得到预测值。

In [ ]:
class TwoLayerNet:
    def __init__(self, input_size=2, hidden_size=4, output_size=1):
        # Xavier 初始化：让权重的方差适配层的大小
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        """前向传播：X -> h -> y_hat"""
        # 第一层
        self.z1 = X @ self.W1 + self.b1     # (N, hidden)
        self.h  = sigmoid(self.z1)           # (N, hidden)
        # 第二层
        self.z2 = self.h @ self.W2 + self.b2 # (N, 1)
        self.y_hat = sigmoid(self.z2)         # (N, 1)
        return self.y_hat

# 测试前向传播（未训练，输出随机）
net = TwoLayerNet()
y_hat = net.forward(X)
print('未训练时的预测（随机）:')
for i, (xi, yi, yh) in enumerate(zip(X, y, y_hat)):
    print(f'  x={xi.astype(int)}, 真值={int(yi[0])}, 预测={yh[0]:.3f}')

## 4. 反向传播（Backward Pass）

这是核心！用**链式法则**，从输出层往回，逐层计算每个权重的梯度。

```
损失 L
  ↓ ∂L/∂ŷ = ŷ - y
输出层权重 W2
  ↓ 链式法则传回
隐藏层权重 W1
```

**关键公式（每步都用了链式法则）**：

1. $\delta_2 = (\hat{y} - y) \cdot \sigma'(z_2)$ — 输出层误差信号
2. $\delta_1 = (\delta_2 \cdot W_2^T) \cdot \sigma'(z_1)$ — 把误差「传」回隐藏层
3. $\nabla W_2 = h^T \cdot \delta_2$，$\nabla W_1 = X^T \cdot \delta_1$ — 计算梯度
4. 梯度下降更新：$W \leftarrow W - \eta \nabla W$

In [ ]:
def backward(self, X, y, lr=0.1):
    """反向传播 + 权重更新"""
    N = X.shape[0]

    # ---- 输出层 ----
    # 损失对 y_hat 的梯度：dL/dyhat = (yhat - y)
    dL_dyhat = self.y_hat - y                    # (N, 1)
    # 链式法则：dL/dz2 = dL/dyhat * sigmoid'(z2)
    delta2 = dL_dyhat * sigmoid_deriv(self.z2)  # (N, 1)

    # W2 的梯度：dL/dW2 = h^T · delta2
    dW2 = self.h.T @ delta2 / N                 # (hidden, 1)
    db2 = delta2.mean(axis=0, keepdims=True)    # (1, 1)

    # ---- 隐藏层 ----
    # 把误差信号反向传回隐藏层：delta2 * W2^T
    delta1 = (delta2 @ self.W2.T) * sigmoid_deriv(self.z1)  # (N, hidden)

    # W1 的梯度：dL/dW1 = X^T · delta1
    dW1 = X.T @ delta1 / N                      # (2, hidden)
    db1 = delta1.mean(axis=0, keepdims=True)    # (1, hidden)

    # ---- 梯度下降更新 ----
    self.W2 -= lr * dW2
    self.b2 -= lr * db2
    self.W1 -= lr * dW1
    self.b1 -= lr * db1

# 把 backward 方法加到类里（monkey-patch，避免重写整个类）
TwoLayerNet.backward = backward
print('反向传播方法已注册')

## 5. 训练循环

不断重复：前向传播 → 计算损失 → 反向传播 → 更新权重。

这就是神经网络「学习」的全部过程。

In [ ]:
def mse_loss(y_hat, y):
    """均方误差损失"""
    return np.mean((y_hat - y) ** 2)

# 重新初始化，保证可复现
np.random.seed(42)
net = TwoLayerNet(hidden_size=4)
TwoLayerNet.backward = backward

EPOCHS = 10000
LR = 0.5
loss_history = []

for epoch in range(EPOCHS):
    # 前向传播
    y_hat = net.forward(X)
    # 计算损失
    loss = mse_loss(y_hat, y)
    loss_history.append(loss)
    # 反向传播 + 更新
    net.backward(X, y, lr=LR)
    # 每 1000 步打印一次
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1:5d} | Loss: {loss:.6f}')

print(f'\n训练完成！最终损失: {loss_history[-1]:.6f}')

In [ ]:
# 绘制训练损失曲线
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='steelblue', linewidth=1.5)
plt.xlabel('训练轮次 (Epoch)')
plt.ylabel('均方误差损失 (MSE)')
plt.title('反向传播训练过程：损失随轮次下降')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/assets/03_loss_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print('损失从初始约 0.25 降至 <0.001，下降了约 250 倍')

## 6. 验证：XOR 终于通了！

17年前，单层感知机在这4个数据点上失败了。现在来看看两层网络表现如何。

In [ ]:
# 用训练好的网络预测
y_pred = net.forward(X)
y_binary = (y_pred > 0.5).astype(int)

print('XOR 预测结果：')
print(f'{"输入":<10} {"真值":<8} {"预测概率":<12} {"预测标签"}')
print('-' * 45)
all_correct = True
for xi, yi, yp, yb in zip(X, y, y_pred, y_binary):
    status = '✓' if yb[0] == int(yi[0]) else '✗'
    print(f'{str(xi.astype(int)):<10} {int(yi[0]):<8} {yp[0]:.4f}       {yb[0]}  {status}')
    if yb[0] != int(yi[0]):
        all_correct = False

accuracy = np.mean(y_binary == y.astype(int)) * 100
print(f'\n准确率: {accuracy:.0f}%')
if all_correct:
    print('XOR 问题完全解决！反向传播成功。')

In [ ]:
# 决策边界可视化
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300),
                     np.linspace(-0.5, 1.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = net.forward(grid).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, probs, levels=20, cmap='RdYlBu_r', alpha=0.8)
plt.colorbar(label='网络输出概率')

# 画出 XOR 四个点
colors = ['blue', 'red']
labels_text = ['XOR=0', 'XOR=1']
for i, (xi, yi) in enumerate(zip(X, y)):
    label = labels_text[int(yi[0])] if i in [0, 3] or i == 1 else '_'
    plt.scatter(xi[0], xi[1],
                color=colors[int(yi[0])],
                s=200, zorder=5,
                label=label if i < 2 else '_')
    plt.annotate(f'({int(xi[0])},{int(xi[1])})\nXOR={int(yi[0])}',
                 xy=(xi[0], xi[1]), xytext=(xi[0]+0.07, xi[1]+0.07),
                 fontsize=9)

plt.contour(xx, yy, probs, levels=[0.5], colors='white', linewidths=2)
plt.title('两层网络的决策边界\n（白线 = 0.5 概率分界）', fontsize=12)
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../docs/assets/03_decision_boundary.png', dpi=100, bbox_inches='tight')
plt.show()
print('注意：决策边界是非线性曲线，这是单层网络做不到的')

## 7. 回顾与思考

这个 notebook 演示了：

1. **链式法则** 是反向传播的数学核心——误差信号可以从输出层逐层往回传
2. **两层网络** 成功解决了 XOR，这在1969年的单层网络上是不可能的
3. **sigmoid 导数最大只有 0.25**——想象网络有 10 层，梯度传到第1层只有原来的 $0.25^{10} \approx 10^{-6}$，这就是**梯度消失**问题

**动手试试**：把 `hidden_size` 改成 2、8、16，观察训练速度和最终精度的变化。

> 下一节点：**LeNet（1989）** — Yann LeCun 把反向传播用到了卷积结构，机器第一次能识别手写数字